# M2/M3 — kiểm train trên GPU

31 cặp đã kiểm; mỗi nhánh chạy 5 update rồi nạp checkpoint-4 và nối một update. Kiểm loss, VRAM, policy/reference và resume. Đây chưa phải thí nghiệm chính.

Chạy 1 → 2 → 3 → 4 → 5. M1 đã xong; không chạy lại các ô lịch sử bên dưới. Không bấm Chạy tất cả.

Log và adapter lưu trong Drive `runs/preference_gpu_gates`. Bộ cặp cuối cần chốt N riêng; run chính khởi đầu lại từ M1 sạch.


In [ ]:
import subprocess
from pathlib import Path

from google.colab import drive

gpu_command = ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']
print(subprocess.check_output(gpu_command, text=True))
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Vietnamese_VQA_NLE')
assert ROOT.is_dir(), 'Project Drive folder is missing'
print('Project:', ROOT)


In [ ]:
import hashlib
import json
import stat
import sys
import zipfile
from tempfile import TemporaryDirectory

launch = json.loads((ROOT / 'source/preference_gpu_gate_v1.json').read_text())
assert launch['scope'] == 'technical_gpu_gate_not_main_experiment'
SOURCE_RELEASE = launch['source_release']
SOURCE_SHA256 = launch['source_sha256']
assert SOURCE_RELEASE.startswith('source_') and '/' not in SOURCE_RELEASE
assert '\\' not in SOURCE_RELEASE and ':' not in SOURCE_RELEASE
archive_path = ROOT / 'source/releases' / f'{SOURCE_RELEASE}.zip'
assert hashlib.sha256(archive_path.read_bytes()).hexdigest() == SOURCE_SHA256
source_parent = Path('/content/vqa_sources') / SOURCE_RELEASE
GATE_REPO = source_parent / 'Vietnamese-VQA-Explanation'
with zipfile.ZipFile(archive_path) as archive:
    manifest = json.loads(archive.read('Vietnamese-VQA-Explanation/SOURCE_MANIFEST.json'))
    expected = {'Vietnamese-VQA-Explanation/' + name for name in manifest}
    expected.add('Vietnamese-VQA-Explanation/SOURCE_MANIFEST.json')
    assert len(archive.namelist()) == len(expected) and set(archive.namelist()) == expected
    assert archive.testzip() is None
    for name, digest in manifest.items():
        assert name and not name.startswith('/') and ':' not in name and '\\' not in name
        assert all(part not in ('', '.', '..') for part in name.split('/'))
        member = archive.getinfo('Vietnamese-VQA-Explanation/' + name)
        assert stat.S_IFMT(member.external_attr >> 16) != stat.S_IFLNK
        assert hashlib.sha256(archive.read(member)).hexdigest() == digest
    if not GATE_REPO.exists():
        assert not source_parent.exists(), 'Incomplete source tree; use a new runtime'
        source_parent.parent.mkdir(parents=True, exist_ok=True)
        with TemporaryDirectory(dir=source_parent.parent) as temporary:
            archive.extractall(temporary)
            Path(temporary).rename(source_parent)
    for name, digest in manifest.items():
        assert hashlib.sha256((GATE_REPO / name).read_bytes()).hexdigest() == digest
install = [sys.executable, '-m', 'pip', 'install', '-q', '-r',
           'requirements-cloud-preference.txt']
subprocess.run(install, cwd=GATE_REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], cwd=GATE_REPO, check=True)
print('Source ready:', SOURCE_RELEASE)


In [ ]:
from datetime import datetime, timezone

session_name = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
GATE_SESSION = ROOT / 'runs/preference_gpu_gates' / session_name
GATE_TOOL = GATE_REPO / 'scripts/run_preference_gpu_gate.py'
command = [sys.executable, str(GATE_TOOL), 'prepare', '--session', str(GATE_SESSION),
    '--inputs', str(ROOT / 'preference_local/preference_gpu_inputs_31_20261013.zip'),
    '--inputs-sha256', '9e64be48cf1202fdf94a3faf3dbd02de894c5c703ed36169f87618c68773080c',
    '--work-dir', '/content/vqa_preference_gate',
    '--m1-adapter', str(ROOT / 'runs/m1/sft-full_20261007_023531_870900/training/checkpoint-3683'),
    '--m1-selection', str(ROOT / 'reports/m1_selection/frozen_checkpoint3683_v2/selection.json')]
subprocess.run(command, cwd=GATE_REPO, check=True)
print('Session:', GATE_SESSION)


In [ ]:
command = [sys.executable, str(GATE_TOOL), 'm2', '--session', str(GATE_SESSION)]
subprocess.run(command, cwd=GATE_REPO, check=True)


In [ ]:
command = [sys.executable, str(GATE_TOOL), 'm3', '--session', str(GATE_SESSION)]
subprocess.run(command, cwd=GATE_REPO, check=True)
command = [sys.executable, str(GATE_TOOL), 'report', '--session', str(GATE_SESSION)]
subprocess.run(command, cwd=GATE_REPO, check=True)


# Lịch sử M1 và sinh ứng viên — không chạy lại

Các ô bên dưới giữ nguyên source và output đã lưu. Số ô trong phần mô tả cũ thuộc phiên M1, không phải thứ tự hiện hành.


# M1 — đã hoàn tất huấn luyện và đánh giá

Train đã hoàn tất 3.683 bước, một epoch trên 29.459 mẫu. Run: `runs/m1/sft-full_20261007_023531_870900`.

Đã chọn checkpoint-3683 bằng loss trên 256 validation cố định. Mốc 1.200 và 3.683 đều đã đánh giá đủ 1.407 câu, cùng protocol và cache M0. Không train hoặc sinh lại prediction.

Đã sinh đủ 2.000 ứng viên train từ M1-3683 ngày 08/10. Bước hiện tại: kiểm ảnh và tạo cặp preference. Chưa có accepted, chưa train M2/M3.

Không chạy lại ô 10: summary đã đủ 2.000. Không bấm Chạy tất cả. Luồng 1 → 9 → 10 chỉ dùng nếu cần khôi phục lượt sinh chưa hoàn tất; ô 2–7 và từ ô 11 giữ lịch sử M1.

Nếu cần đọc lại kết quả trên VM mới: chạy ô 1 và 6–7; restart do ô 6 thì chỉ chạy lại ô 1 và 6–7, không chạy ô 2. Không dùng test để chọn checkpoint.


In [ ]:
import json
from pathlib import Path

import torch
from google.colab import drive

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive', timeout_ms=600000)
ROOT = Path('/content/drive/MyDrive/Vietnamese_VQA_NLE')
assert ROOT.is_dir(), f'Sai tài khoản Drive hoặc thiếu thư mục: {ROOT}'
print('Drive:', ROOT)

# Chỉ đọc trạng thái đã lưu; không tự chạy lại hoặc chọn checkpoint.
RUN_ROOT = ROOT / 'runs'
for filename in ('token_full_M1_v1.json', 'm1_environment_v1.json'):
    saved = RUN_ROOT / filename
    if saved.is_file():
        payload = json.loads(saved.read_text(encoding='utf-8'))
        print(filename, json.dumps(payload.get('summary', payload), ensure_ascii=False))
    else:
        print(filename, 'chưa có')
executions = [*RUN_ROOT.glob('sft-full_*/execution.json'),
              *(RUN_ROOT / 'm1').glob('sft-full_*/execution.json')]
for execution in sorted(executions, key=lambda p: p.parent.name)[-3:]:
    payload = json.loads(execution.read_text(encoding='utf-8'))
    print('M1 run:', execution.parent.relative_to(ROOT),
          'status:', payload.get('status'), 'budget:', payload.get('budget'))
    heartbeat = execution.parent / 'runner_heartbeat.json'
    if heartbeat.is_file():
        print('Heartbeat:', heartbeat.read_text(encoding='utf-8'))
    for saved in sorted((execution.parent / 'training').glob('checkpoint-*/trainer_state.json')):
        state = json.loads(saved.read_text(encoding='utf-8'))
        print('Saved checkpoint:', saved.parent.name, 'step:', state.get('global_step'),
              'epoch:', state.get('epoch'), 'best loss:', state.get('best_metric'))
    log = execution.parent / 'console.log'
    if log.is_file():
        with log.open('rb') as stream:
            stream.seek(max(0, log.stat().st_size - 1800))
            print('Log tail:', stream.read().decode('utf-8', errors='replace'))


In [ ]:
import hashlib
import importlib.metadata
import stat
import subprocess
import sys
import zipfile
from tempfile import TemporaryDirectory

SOURCE_RELEASE = 'source_20260928_051956_295773'
EXPECTED_SOURCE_SHA256 = 'b437eb01e8a01b1e614fc31b295096c020dcc20b37b5a4644344b6c95a5891d0'
SOURCE_DIR = ROOT / 'source/releases'
SOURCE_HASH = SOURCE_DIR / f'{SOURCE_RELEASE}.sha256'
SOURCE_ZIP = SOURCE_DIR / f'{SOURCE_RELEASE}.zip'
if not SOURCE_HASH.is_file() or not SOURCE_ZIP.is_file():
    raise FileNotFoundError(f'Thiếu source release đã ghim: {SOURCE_HASH} hoặc {SOURCE_ZIP}')
checksum_parts = SOURCE_HASH.read_text(encoding='utf-8').split()
if len(checksum_parts) != 2 or checksum_parts != [EXPECTED_SOURCE_SHA256, SOURCE_ZIP.name]:
    raise ValueError(f'Checksum manifest không khớp release đã ghim: {SOURCE_HASH}')
digest = hashlib.sha256()
with SOURCE_ZIP.open('rb') as source:
    for block in iter(lambda: source.read(1024 * 1024), b''):
        digest.update(block)
if digest.hexdigest() != EXPECTED_SOURCE_SHA256:
    raise ValueError(f'Source ZIP checksum mismatch: {SOURCE_ZIP}')

ARCHIVE_ROOT = 'Vietnamese-VQA-Explanation/'
MANIFEST_MEMBER = ARCHIVE_ROOT + 'SOURCE_MANIFEST.json'

def _sha256_stream(source):
    digest = hashlib.sha256()
    for block in iter(lambda: source.read(1024 * 1024), b''):
        digest.update(block)
    return digest.hexdigest()

def _verify_source_archive(archive):
    members = archive.infolist()
    names = [member.filename for member in members]
    if len(names) != len(set(names)) or MANIFEST_MEMBER not in names:
        raise ValueError('Source ZIP has duplicate or missing manifest members')
    manifest_bytes = archive.read(MANIFEST_MEMBER)
    manifest = json.loads(manifest_bytes)
    if not isinstance(manifest, dict) or not manifest:
        raise ValueError('Source manifest must be a nonempty object')
    for relative, expected in manifest.items():
        if (not isinstance(relative, str) or not relative
                or any(part in ('', '.', '..') for part in relative.split('/'))
                or '\\' in relative or ':' in relative):
            raise ValueError(f'Unsafe source path: {relative!r}')
        if (not isinstance(expected, str) or len(expected) != 64
                or any(char not in '0123456789abcdef' for char in expected)):
            raise ValueError(f'Invalid source hash: {relative}')
    expected_names = {ARCHIVE_ROOT + relative for relative in manifest}
    expected_names.add(MANIFEST_MEMBER)
    if set(names) != expected_names or any(
        member.is_dir() or stat.S_IFMT(member.external_attr >> 16) == stat.S_IFLNK
        for member in members
    ):
        raise ValueError('Source ZIP members do not match safe manifest files')
    if archive.testzip() is not None:
        raise ValueError('Source ZIP failed CRC check')
    for relative, expected in manifest.items():
        with archive.open(ARCHIVE_ROOT + relative) as source:
            if _sha256_stream(source) != expected:
                raise ValueError(f'Source file hash mismatch: {relative}')
    return manifest, hashlib.sha256(manifest_bytes).hexdigest()

def _restore_build_metadata(repo, archive, manifest):
    # Editable installs regenerate egg-info; restore it from the verified ZIP.
    for relative in manifest:
        parts = Path(relative).parts
        if not any(part.endswith('.egg-info') for part in parts):
            continue
        file = repo / relative
        if any((repo / Path(*parts[:index])).is_symlink()
               for index in range(1, len(parts) + 1)) or not file.is_file():
            raise ValueError(f'Unsafe build metadata path: {relative}')
        file.write_bytes(archive.read(ARCHIVE_ROOT + relative))

def _verify_source_tree(repo, manifest, manifest_hash):
    manifest_file = repo / 'SOURCE_MANIFEST.json'
    if not manifest_file.is_file() or manifest_file.is_symlink():
        raise ValueError('Source tree has no regular manifest file')
    with manifest_file.open('rb') as source:
        if _sha256_stream(source) != manifest_hash:
            raise ValueError('Source tree manifest differs from ZIP')
    for relative, expected in manifest.items():
        file = repo / relative
        if not file.is_file() or file.is_symlink():
            raise ValueError(f'Source tree file missing or linked: {relative}')
        with file.open('rb') as source:
            if _sha256_stream(source) != expected:
                raise ValueError(f'Source tree file hash mismatch: {relative}')
    executable = set()
    for folder in ('src', 'scripts', 'configs'):
        for file in (repo / folder).rglob('*'):
            relative = file.relative_to(repo)
            if (file.suffix in ('.py', '.yaml', '.yml')
                    and not any(part == '__pycache__' or part.endswith('.egg-info')
                                for part in relative.parts)):
                executable.add(relative.as_posix())
    executable.update(file.name for file in repo.glob('*.py'))
    unexpected = executable - manifest.keys()
    if unexpected:
        raise ValueError(f'Unexpected executable source files: {sorted(unexpected)}')
WORK_SOURCE = Path('/content/vqa_sources') / SOURCE_HASH.stem
REPO = WORK_SOURCE / 'Vietnamese-VQA-Explanation'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    manifest, manifest_hash = _verify_source_archive(archive)
    if REPO.exists():
        _restore_build_metadata(REPO, archive, manifest)
        _verify_source_tree(REPO, manifest, manifest_hash)
    else:
        if WORK_SOURCE.exists():
            raise FileExistsError(f'Incomplete source tree; use a clean runtime: {WORK_SOURCE}')
        WORK_SOURCE.parent.mkdir(parents=True, exist_ok=True)
        with TemporaryDirectory(prefix='source-stage-', dir=WORK_SOURCE.parent) as stage:
            archive.extractall(stage)
            _verify_source_tree(Path(stage) / 'Vietnamese-VQA-Explanation',
                                manifest, manifest_hash)
            Path(stage).rename(WORK_SOURCE)
        _verify_source_tree(REPO, manifest, manifest_hash)
assert (REPO / 'scripts/train_sft_full.py').is_file(), 'Release thiếu runner M1'

def _restore_m1_packages():
    expected = {'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128',
                'transformers': '4.57.6', 'peft': '0.20.0', 'accelerate': '1.14.0',
                'bitsandbytes': '0.50.2', 'pillow': '11.3.0'}
    installed = {}
    for name in expected:
        try:
            installed[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            installed[name] = None
    changed = [name for name in expected if installed[name] != expected[name]]
    if not changed:
        return
    print('Restoring M1 packages:', {name: installed[name] for name in changed}, flush=True)
    if any(name in changed for name in ('torch', 'torchvision')):
        subprocess.run([sys.executable, '-m', 'pip', 'install',
                        'torch==2.11.0+cu128', 'torchvision==0.26.0+cu128',
                        '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
    other = [f'{name}=={expected[name]}' for name in changed
             if name not in ('torch', 'torchvision')]
    if other:
        subprocess.run([sys.executable, '-m', 'pip', 'install', *other], check=True)
    raise RuntimeError('M1 packages restored. Restart Python, then rerun cells 1–2.')

_restore_m1_packages()
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[train]'],
    check=True,
)
print('Source verified:', SOURCE_RELEASE, EXPECTED_SOURCE_SHA256)

In [ ]:
BUNDLE = ROOT / 'datasets/releases/vivqax_image_disjoint_v1'
EXPECTED_BUNDLE_SHA256 = '46846a81c7b3594d4fc9670ec51019cfd652f1eb169af7978e39dc5a40a55522'
WORK_DATA = Path('/content/vqa_work/vivqax_image_disjoint_v1')
assert hashlib.sha256((BUNDLE / 'bundle.json').read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256
bundle = json.loads((BUNDLE / 'bundle.json').read_text(encoding='utf-8'))
command = [sys.executable, str(REPO / 'scripts/verify_coco_bundle.py'), str(BUNDLE)]
if not WORK_DATA.exists():
    command += ['--extract-to', str(WORK_DATA)]
subprocess.run(command, cwd=REPO, check=True)
assert len(list((WORK_DATA / 'images').glob('*.jpg'))) == bundle['total_images']
checksums = WORK_DATA / 'metadata/full_image_checksums.csv'
assert hashlib.sha256(checksums.read_bytes()).hexdigest() == bundle['image_checksums_sha256']
assert (WORK_DATA / 'metadata/all.jsonl').is_file()
print('Data verified:', bundle['total_images'], 'images')

In [ ]:
# Verify the completed epoch; no training or checkpoint writes.
import hashlib
import importlib.util
import json
from importlib import metadata as installed_metadata


def download_epoch_report(path):
    from google.colab import files

    files.download(str(path))

assert 'ROOT' in globals() and ROOT.is_dir(), 'Run cell 1 to mount Drive.'
finished_run = ROOT / 'runs/m1/sft-full_20261007_023531_870900'
training = finished_run / 'training'
evidence_paths = [
    finished_run / 'execution.json',
    *[training / name for name in ('training_result.json', 'trainer_state.json',
                                   'data_plan.json', 'run_metadata.json')],
]
before_hashes = {str(path.relative_to(finished_run)): hashlib.sha256(path.read_bytes()).hexdigest()
                 for path in evidence_paths}
execution = json.loads(evidence_paths[0].read_text(encoding='utf-8'))
assert execution['stage'] == 'sft-full' and execution['status'] == 'passed', execution['status']
assert type(execution['returncode']) is int and execution['returncode'] == 0
assert execution['budget']['protocol_id'] == 'vqa_nle_m1_epoch1_v1'
assert execution['budget']['completion_mode'] == 'full_epoch'
assert execution['budget']['stop_after_step'] == 3683

helper_sha = 'c9646355195f940e9cac5a6d63818f6df4c7da591085cea773bc01534ed8256d'
helper = ROOT / 'source' / f'resume_pinned_m1_{helper_sha[:12]}.py'
assert hashlib.sha256(helper.read_bytes()).hexdigest() == helper_sha, 'Completion checker changed.'
spec = importlib.util.spec_from_file_location('m1_epoch_checker', helper)
checker = importlib.util.module_from_spec(spec)
spec.loader.exec_module(checker)
completion = checker.verify_epoch_completion(finished_run, 3683)
integrity = checker.verify_checkpoint_payloads(training / 'checkpoint-3683')
assert execution['epoch_completion'] == completion, 'Recorded completion differs.'
assert execution['output_checkpoint_integrity'] == integrity, 'Final checkpoint differs.'

result = json.loads((training / 'training_result.json').read_text(encoding='utf-8'))
state = json.loads((training / 'trainer_state.json').read_text(encoding='utf-8'))
plan = json.loads((training / 'data_plan.json').read_text(encoding='utf-8'))
metadata = json.loads((training / 'run_metadata.json').read_text(encoding='utf-8'))
assert result['optimizer_steps_this_run'] == 3683 - result['resumed_from_global_step']
assert state['num_train_epochs'] == 1 and metadata['status'] == 'training_passed'
assert (plan['train_count'], plan['validation_count'], plan['test_count'],
        plan['checkpoint_validation_count']) == (29459, 1407, 1968, 256)
assert not any(plan['split_overlap'].values()), 'Image overlap between splits.'
versions = {name: installed_metadata.version(name) for name in metadata['versions']}
assert versions == metadata['versions'], 'Runtime versions differ.'
assert before_hashes == {
    str(path.relative_to(finished_run)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in evidence_paths
}, 'Run metadata changed during verification.'

epoch_report = {
    'status': 'passed', 'run': str(finished_run.relative_to(ROOT)),
    'execution': {key: execution[key] for key in
                  ('stage', 'status', 'returncode', 'started_at_utc', 'finished_at_utc', 'budget')},
    'training_result': result, 'run_metadata': metadata, 'data_plan': plan,
    'epoch_completion': completion, 'final_checkpoint_integrity': integrity,
    'validation_losses': [row for row in state['log_history'] if 'eval_loss' in row],
    'evidence_file_sha256': before_hashes, 'checker_sha256': helper_sha,
    'scope': 'Training/state integrity only; inference quality not yet evaluated.',
}
report_text = json.dumps(epoch_report, ensure_ascii=False, indent=2, sort_keys=True) + '\n'
report_path = ROOT / 'reports/checkpoints' / f'm1_epoch_{finished_run.name}.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_is_new = not report_path.exists()
if not report_is_new:
    assert report_path.read_text(encoding='utf-8') == report_text, 'Existing epoch report differs.'
else:
    with report_path.open('x', encoding='utf-8') as stream:
        stream.write(report_text)
print('M1 epoch verified:', completion['final_optimizer_step'], '| epoch:', completion['epoch'])
print('Trainer selected:', completion['selected_step'], '| loss:', completion['selected_eval_loss'])
print('Final payload:', integrity['global_step'], '| dtype:', integrity['compute_dtype'],
      '| scheduler:', integrity['scheduler_step'],
      '| adapter tensors:', integrity['adapter_tensors'])
print('Audit:', report_path, hashlib.sha256(report_path.read_bytes()).hexdigest())
if report_is_new:
    download_epoch_report(report_path)


In [ ]:
# Measure the final checkpoint with the frozen M1 loss protocol.
import hashlib
import json
import subprocess
import sys
from pathlib import Path

assert 'ROOT' in globals() and ROOT.is_dir(), 'Run cell 1 to mount Drive.'
assert 'REPO' in globals() and REPO.is_dir(), 'Run setup cells 2 and 3.'
EVALUATOR_SHA = '062725a979f6fdc4e673a4471ef2079b8c22b019846a23cc83eede3896e13e03'
CHECKER_SHA = 'c9646355195f940e9cac5a6d63818f6df4c7da591085cea773bc01534ed8256d'
EPOCH_TOOLS = ROOT / 'source/evaluation' / EVALUATOR_SHA[:12]
EPOCH_TOOLS.mkdir(parents=True, exist_ok=True)
for name, expected in (('evaluate_m1_epoch.py', EVALUATOR_SHA),
                       ('resume_pinned_m1.py', CHECKER_SHA)):
    destination = EPOCH_TOOLS / name
    if destination.exists():
        payload = destination.read_bytes()
        assert hashlib.sha256(payload).hexdigest() == expected, f'Changed tool: {name}'
    else:
        uploaded = Path('/content') / name
        assert uploaded.is_file(), f'Upload scripts/{name} to the Colab Files pane.'
        payload = uploaded.read_bytes()
        assert hashlib.sha256(payload).hexdigest() == expected, f'Wrong upload: {name}'
        with destination.open('xb') as stream:
            stream.write(payload)

FINISHED_M1 = ROOT / 'runs/m1/sft-full_20261007_023531_870900'
EPOCH_SELECTION = ROOT / 'reports/m1_selection/epoch_sft-full_20261007_023531_870900'
selection_path = EPOCH_SELECTION / 'selection.json'
selection_is_new = not EPOCH_SELECTION.exists()
if selection_is_new:
    assert torch.cuda.is_available(), 'GPU required; do not evaluate on CPU.'
    EPOCH_SELECTION.parent.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, '-u', str(EPOCH_TOOLS / 'evaluate_m1_epoch.py'),
               '--source', str(REPO),
               '--environment', str(REPO / 'configs/environments/colab_full.example.yaml'),
               '--run', str(FINISHED_M1), '--output-dir', str(EPOCH_SELECTION)]
    print('Checkpoint-3683: 256 fixed validation; no training or test.', flush=True)
    with EPOCH_SELECTION.with_suffix('.log').open('x', encoding='utf-8') as log:
        with subprocess.Popen(command, cwd=EPOCH_TOOLS, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8') as process:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end='', flush=True)
            assert process.wait() == 0, 'Epoch evaluation failed; inspect the saved log.'
selection = json.loads(selection_path.read_text(encoding='utf-8'))
assert selection['status'] == 'passed', 'Inspect this job before creating another.'
assert selection['validation_count'] == 256 and selection['test_predictions_generated'] == 0
assert Path(selection['run']).resolve() == FINISHED_M1.resolve()
assert selection['inputs']['evaluator_sha256'] == EVALUATOR_SHA
assert selection['inputs']['checkpoint_checker_sha256'] == CHECKER_SHA
selected_weights = Path(selection['selected_adapter']) / 'adapter_model.safetensors'
assert hashlib.sha256(selected_weights.read_bytes()).hexdigest() == (
    selection['selected_adapter_sha256'])
print('Trainer checkpoint loss:', selection['trainer_selected_loss'])
print('Checkpoint-3683 loss:', selection['final_checkpoint_loss'])
print('Selected step:', selection['selected_step'], '| validation:', selection['validation_count'])
print('Selection:', selection_path, hashlib.sha256(selection_path.read_bytes()).hexdigest())
if selection_is_new:
    from google.colab import files

    files.download(str(selection_path))


In [ ]:
# Restore the frozen inference source and the M0 cache environment.
import hashlib
import json
import os
import stat as archive_stat
import subprocess
import sys
import zipfile
from importlib import metadata as eval_packages
from pathlib import Path
from tempfile import TemporaryDirectory

assert 'ROOT' in globals() and ROOT.is_dir(), 'Run cell 1 to mount Drive.'
selection_file = ROOT / 'reports/m1_selection/epoch_sft-full_20261007_023531_870900/selection.json'
selected_m1 = json.loads(selection_file.read_text(encoding='utf-8'))
assert selected_m1['status'] == 'passed' and selected_m1['validation_count'] == 256
assert selected_m1['test_predictions_generated'] == 0

EVAL_RELEASE = 'source_20261002_062529_455947'
EVAL_SHA256 = '6b00d2d0297d06f4365248fd2bf63fd31c69808700888219ffd6e7c90283d9fb'
eval_zip = ROOT / 'source/releases' / f'{EVAL_RELEASE}.zip'
assert hashlib.sha256(eval_zip.read_bytes()).hexdigest() == EVAL_SHA256
EVAL_BASE = Path('/content/vqa_eval_sources') / EVAL_RELEASE
EVAL_REPO = EVAL_BASE / 'Vietnamese-VQA-Explanation'
prefix = 'Vietnamese-VQA-Explanation/'
with zipfile.ZipFile(eval_zip) as archive:
    entries = archive.infolist()
    names = [entry.filename for entry in entries]
    manifest_bytes = archive.read(prefix + 'SOURCE_MANIFEST.json')
    eval_manifest = json.loads(manifest_bytes)
    assert isinstance(eval_manifest, dict) and eval_manifest
    assert len(names) == len(set(names))
    expected_names = {prefix + name for name in eval_manifest}
    assert set(names) == expected_names | {prefix + 'SOURCE_MANIFEST.json'}
    for name, expected in eval_manifest.items():
        assert name and ':' not in name and '\\' not in name
        assert all(part not in ('', '.', '..') for part in name.split('/'))
        assert hashlib.sha256(archive.read(prefix + name)).hexdigest() == expected
    assert all(not entry.is_dir() and archive_stat.S_IFMT(entry.external_attr >> 16)
               != archive_stat.S_IFLNK
               for entry in entries)
    if not EVAL_REPO.exists():
        EVAL_BASE.mkdir(parents=True, exist_ok=True)
        with TemporaryDirectory(dir=EVAL_BASE) as temporary:
            archive.extractall(temporary)
            (Path(temporary) / 'Vietnamese-VQA-Explanation').rename(EVAL_REPO)
assert not EVAL_REPO.is_symlink()
assert (EVAL_REPO / 'SOURCE_MANIFEST.json').read_bytes() == manifest_bytes
for name, expected in eval_manifest.items():
    file = EVAL_REPO / name
    assert file.is_file() and not file.is_symlink()
    assert hashlib.sha256(file.read_bytes()).hexdigest() == expected, name

def verified_m0_cache(project, uploads=Path('/content')):
    restored = project / 'evaluation_runs/m0_full_validation_cache_v1/m0.jsonl'
    candidates = [restored,
                  project / 'evaluation_runs/baseline_validation_600_mount_v1/m0.jsonl',
                  project / 'evaluation_local/baseline_validation_800_v1/m0.jsonl']
    expected = ('0e7fb9d4b2501d0a65ee161de61c12fef5a0ad511d16190dbdd368792c3db97d',
                'd03fb3ef07accd316aa324190fcf0ac7978a3c5e93922ac2718cf6f7545dcba0')
    for predictions in candidates:
        manifest = predictions.with_suffix('.jsonl.manifest.json')
        if not predictions.is_file() or not manifest.is_file():
            continue
        actual = tuple(hashlib.sha256(file.read_bytes()).hexdigest()
                       for file in (predictions, manifest))
        assert actual == expected, f'M0 cache changed: {predictions}'
        return predictions
    uploaded = [uploads / name for name in ('m0.jsonl', 'm0.jsonl.manifest.json')]
    if not all(file.is_file() for file in uploaded):
        raise FileNotFoundError('No verified M0 cache accessible. Upload the saved '
                                'm0.jsonl and its manifest through the Files pane.')
    payloads = [file.read_bytes() for file in uploaded]
    assert tuple(hashlib.sha256(data).hexdigest() for data in payloads) == expected
    restored.parent.mkdir(parents=True, exist_ok=True)
    for name, data in zip(('m0.jsonl', 'm0.jsonl.manifest.json'), payloads, strict=True):
        destination = restored.parent / name
        if destination.exists():
            assert destination.read_bytes() == data, f'Existing M0 cache differs: {name}'
        else:
            with destination.open('xb') as stream:
                stream.write(data)
        assert destination.read_bytes() == data
    return restored

M0_CACHE = verified_m0_cache(ROOT)
cache_manifest = json.loads(
    M0_CACHE.with_suffix('.jsonl.manifest.json').read_text(encoding='utf-8'))
expected_versions = {'torch': '2.11.0+cu130', 'transformers': '4.57.6',
                     'peft': '0.21.0', 'bitsandbytes': '0.50.2', 'pillow': '11.3.0'}
assert cache_manifest['inputs']['versions'] == expected_versions
assert cache_manifest['inputs']['python'] == sys.version.split()[0]
for name, expected in cache_manifest['inputs']['source_files'].items():
    assert hashlib.sha256((EVAL_REPO / name).read_bytes()).hexdigest() == expected, name
versions = {name: eval_packages.version(name) for name in expected_versions}
if versions != expected_versions:
    if versions['torch'] != expected_versions['torch']:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        '--index-url', 'https://download.pytorch.org/whl/cu130',
                        'torch==2.11.0+cu130', 'torchvision==0.26.0+cu130'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    *[f'{name}=={version}' for name, version in expected_versions.items()
                      if name != 'torch']], check=True)
    raise RuntimeError('Inference packages restored. Restart Python; '
                       'rerun cell 1, then 6–7, not cell 2.')
assert torch.__version__ == expected_versions['torch'], 'Restart Python before inference.'
assert torch.cuda.is_available(), 'GPU required; do not infer on CPU.'
EVAL_ENV = os.environ.copy()
EVAL_ENV['PYTHONPATH'] = os.pathsep.join([str(EVAL_REPO / 'src'), str(EVAL_REPO)])
print('Inference source verified:', EVAL_RELEASE)
print('Inference environment:', versions)
print('M0 cache:', M0_CACHE, '| selected M1 step:', selected_m1['selected_step'])


In [ ]:
# Evaluate the selected M1 and step 1200 with the same frozen protocol.
import hashlib
import json
import subprocess
import sys
from pathlib import Path

assert 'EVAL_REPO' in globals() and 'EVAL_ENV' in globals(), 'Run cell 6 first.'
assert {name: eval_packages.version(name) for name in expected_versions} == expected_versions
assert torch.__version__ == expected_versions['torch'] and torch.cuda.is_available()
selected_m1 = json.loads(selection_file.read_text(encoding='utf-8'))
assert selected_m1['status'] == 'passed' and selected_m1['validation_count'] == 256
assert selected_m1['inputs']['evaluator_sha256'] == (
    '062725a979f6fdc4e673a4471ef2079b8c22b019846a23cc83eede3896e13e03')
assert selected_m1['test_predictions_generated'] == 0
SELECTED_ADAPTER = Path(selected_m1['selected_adapter']).resolve(strict=True)
selected_weights = SELECTED_ADAPTER / 'adapter_model.safetensors'
assert hashlib.sha256(selected_weights.read_bytes()).hexdigest() == (
    selected_m1['selected_adapter_sha256'])
VALIDATION_RUNNER_SHA = 'dc70200ed0a328c8456fb66f785dd6737666eb6b079b0ee7434953c9e670160a'
VALIDATION_TOOLS = ROOT / 'source/evaluation' / VALIDATION_RUNNER_SHA[:12]
VALIDATION_TOOLS.mkdir(parents=True, exist_ok=True)
VALIDATION_RUNNER = VALIDATION_TOOLS / 'run_baseline_validation.py'
if not VALIDATION_RUNNER.exists():
    uploaded = Path('/content/run_baseline_validation.py')
    assert uploaded.is_file(), 'Upload scripts/run_baseline_validation.py through the Files pane.'
    payload = uploaded.read_bytes()
    assert hashlib.sha256(payload).hexdigest() == VALIDATION_RUNNER_SHA
    with VALIDATION_RUNNER.open('xb') as stream:
        stream.write(payload)
assert hashlib.sha256(VALIDATION_RUNNER.read_bytes()).hexdigest() == VALIDATION_RUNNER_SHA
VALIDATION_JOB = ROOT / 'evaluation_runs' / (
    f'm1_epoch_sft-full_20261007_023531_870900_step{selected_m1["selected_step"]}_v1')

def evaluate_validation(adapter, expected_sha256, job, label):
    weights = adapter / 'adapter_model.safetensors'
    assert hashlib.sha256(weights.read_bytes()).hexdigest() == expected_sha256
    comparison_file = job / 'full_validation.json'
    if not comparison_file.exists():
        command = [sys.executable, '-u', str(VALIDATION_RUNNER),
                   '--environment', str(EVAL_REPO / 'configs/environments/colab_full.example.yaml'),
                   '--checkpoint', str(adapter), '--output-dir', str(job),
                   '--m0-predictions', str(M0_CACHE), '--resume']
        print(f'{label}: 1.407 validation questions; cached M0; no test.', flush=True)
        with subprocess.Popen(command, cwd=EVAL_REPO, env=EVAL_ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8') as process:
            for line in process.stdout:
                print(line, end='', flush=True)
            assert process.wait() == 0, 'Validation stopped; inspect console.log before resuming.'
    comparison = json.loads(comparison_file.read_text(encoding='utf-8'))
    assert comparison['comparable'] and comparison['canonical_references_checked']
    assert comparison['run_identity_metadata_checked'] and comparison['n'] == 1407
    assert comparison['expected_split'] == 'validation' and not comparison['issues']
    assert comparison['explanation_rouge_l_f1']['comparable']
    prediction_manifest = json.loads(
        (job / 'm1.jsonl.manifest.json').read_text(encoding='utf-8'))
    inputs = prediction_manifest['inputs']
    assert inputs['adapter'] == str(adapter.resolve())
    assert inputs['adapter_files']['adapter_model.safetensors'] == expected_sha256
    assert inputs['versions'] == expected_versions
    for field in ('annotations_sha256', 'selected_sha256', 'image_hashes', 'source_files'):
        assert inputs[field] == cache_manifest['inputs'][field], field
    assert prediction_manifest['protocol'] == cache_manifest['protocol']
    predictions = [json.loads(line) for line in
                   (job / 'm1.jsonl').read_text(encoding='utf-8').splitlines()]
    assert len(predictions) == len({row['question_id'] for row in predictions}) == 1407
    assert {row['question_id'] for row in predictions} == set(inputs['image_hashes'])
    assert hashlib.sha256(weights.read_bytes()).hexdigest() == expected_sha256
    print(label, '| questions:', len(predictions),
          '| invalid format:', sum(not row['format_valid'] for row in predictions))
    print('Answer scores:', comparison['scores']['right'])
    print('Explanation ROUGE-L:', comparison['explanation_rouge_l_f1']['scores']['right'])
    print('Full validation report:', comparison_file)
    return comparison

selected_comparison = evaluate_validation(
    SELECTED_ADAPTER, selected_m1['selected_adapter_sha256'], VALIDATION_JOB, 'M1 step 3683')

def verified_checkpoint_1200(project):
    run = project / 'runs/m1/sft-full_20261004_135823_877263'
    execution = json.loads((run / 'execution.json').read_text(encoding='utf-8'))
    assert execution['stage'] == 'sft-full' and execution['status'] == 'paused'
    assert type(execution['returncode']) is int and execution['returncode'] == 0
    integrity = execution['output_checkpoint_integrity']
    assert integrity['global_step'] == integrity['scheduler_step'] == 1200
    checkpoint = (run / 'training/checkpoint-1200').resolve(strict=True)
    for name in ('adapter_config.json', 'adapter_model.safetensors', 'trainer_state.json'):
        assert hashlib.sha256((checkpoint / name).read_bytes()).hexdigest() == (
            integrity['file_sha256'][name]), f'Checkpoint-1200 changed: {name}'
    state = json.loads((checkpoint / 'trainer_state.json').read_text(encoding='utf-8'))
    assert state['global_step'] == 1200 and state['max_steps'] == 3683
    return checkpoint, integrity['file_sha256']['adapter_model.safetensors']

REFERENCE_ADAPTER, REFERENCE_SHA256 = verified_checkpoint_1200(ROOT)
REFERENCE_JOB = ROOT / 'evaluation_runs/m1_checkpoint1200_sft-full_20261004_135823_877263_v1'
print('Reference checkpoint-1200 SHA-256:', REFERENCE_SHA256, flush=True)
reference_comparison = evaluate_validation(
    REFERENCE_ADAPTER, REFERENCE_SHA256, REFERENCE_JOB, 'M1 step 1200')


## Chuẩn bị dữ liệu M2/M3

Đã hoàn tất 2.000 ứng viên từ pool train cố định. Local đã kiểm hash, đủ ID/ảnh và adapter M1; EM 70,15% ở đây không phải điểm test hoặc tỷ lệ cặp sạch.

Duyệt tại `artifacts/preference_editor_m1_3683_2000_20261008/index.html` trên local. Kiểm 30 câu đầu theo ảnh trước; ghi chú AI nằm riêng, không phải nhãn người. Chỉ đóng băng cặp sau khi duyệt thật. Không chạy lại ô 10 hoặc train M1.

Ứng viên chưa duyệt không phải cặp DPO. Output ở `preference_local/candidates_m1_3683_pool2000_v1`; không ghi đè calibration checkpoint-800. Source của giai đoạn này ghim qua `source/preference_stage_v2.json`.


In [ ]:
# Pin M1 and validation evidence; this cell does not need a GPU.
import hashlib
import json
import os
import stat as preference_archive_stat
import subprocess
import sys
import zipfile
from pathlib import Path

assert 'ROOT' in globals() and ROOT.is_dir(), 'Run cell 1 to mount Drive.'
stage_spec = ROOT / 'source/preference_stage_v2.json'
preference_spec = json.loads(stage_spec.read_text(encoding='utf-8'))
assert preference_spec['stage'] == 'm1_preference_candidates_v2'
PREFERENCE_RELEASE = preference_spec['source_release']
assert PREFERENCE_RELEASE.startswith('source_')
assert Path(PREFERENCE_RELEASE).name == PREFERENCE_RELEASE
source_zip = ROOT / 'source/releases' / (PREFERENCE_RELEASE + '.zip')
assert hashlib.sha256(source_zip.read_bytes()).hexdigest() == preference_spec['source_sha256']
PREFERENCE_BASE = Path('/content/vqa_preference_sources') / PREFERENCE_RELEASE
PREFERENCE_REPO = PREFERENCE_BASE / 'Vietnamese-VQA-Explanation'
prefix = 'Vietnamese-VQA-Explanation/'
with zipfile.ZipFile(source_zip) as archive:
    source_manifest = json.loads(archive.read(prefix + 'SOURCE_MANIFEST.json'))
    expected_members = {prefix + name for name in source_manifest}
    assert set(archive.namelist()) == expected_members | {prefix + 'SOURCE_MANIFEST.json'}
    assert len(archive.namelist()) == len(set(archive.namelist()))
    assert archive.testzip() is None
    for name, digest in source_manifest.items():
        assert name and ':' not in name and '\\' not in name
        assert all(part not in ('', '.', '..') for part in name.split('/'))
        assert hashlib.sha256(archive.read(prefix + name)).hexdigest() == digest
    assert all(preference_archive_stat.S_IFMT(entry.external_attr >> 16)
               != preference_archive_stat.S_IFLNK for entry in archive.infolist())
    if not PREFERENCE_REPO.exists():
        archive.extractall(PREFERENCE_REPO.parent)
assert not PREFERENCE_REPO.is_symlink()
for name, digest in source_manifest.items():
    path = PREFERENCE_REPO / name
    assert path.is_file() and not path.is_symlink()
    assert path.resolve().is_relative_to(PREFERENCE_REPO.resolve())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, name
PREFERENCE_ENV = os.environ.copy()
PREFERENCE_ENV['PYTHONPATH'] = os.pathsep.join([str(PREFERENCE_REPO / 'src'), str(PREFERENCE_REPO)])

BUNDLE = ROOT / 'datasets/releases/vivqax_image_disjoint_v1'
assert hashlib.sha256((BUNDLE / 'bundle.json').read_bytes()).hexdigest() == (
    '46846a81c7b3594d4fc9670ec51019cfd652f1eb169af7978e39dc5a40a55522')
bundle = json.loads((BUNDLE / 'bundle.json').read_text(encoding='utf-8'))
metadata_archive = BUNDLE / 'annotations_image_disjoint_v1.zip'
assert hashlib.sha256(metadata_archive.read_bytes()).hexdigest() == (
    bundle['archives'][metadata_archive.name]['sha256'])
canonical = Path('/content/vqa_preference_metadata_v2/all.jsonl')
FROZEN_M1 = ROOT / 'reports/m1_selection/frozen_checkpoint3683_v2/selection.json'
validation_snapshot = FROZEN_M1.parent / 'validation.jsonl'
canonical_sha = '9750ab17cf6dae74680404e167ab1c29d1e2083c8baa2e25bfafcf14bff29c31'
with zipfile.ZipFile(metadata_archive) as archive:
    for name, destination, expected in (
        ('all.jsonl', canonical, canonical_sha),
        ('validation.jsonl', validation_snapshot,
         '7f0b2b7d374d386030ddea50bc57d3e8e6c76e257e9fc4d1203bfb331b5f5b6e'),
    ):
        payload = archive.read('metadata/' + name)
        assert hashlib.sha256(payload).hexdigest() == expected
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists():
            assert destination.read_bytes() == payload, 'Existing annotation snapshot differs.'
        else:
            with destination.open('xb') as stream:
                stream.write(payload)

FINISHED_M1 = ROOT / 'runs/m1/sft-full_20261007_023531_870900'
SELECTED_ADAPTER = FINISHED_M1 / 'training/checkpoint-3683'
VALIDATION_JOB = ROOT / 'evaluation_runs/m1_epoch_sft-full_20261007_023531_870900_step3683_v1'
LOSS_SELECTION = ROOT / 'reports/m1_selection/epoch_sft-full_20261007_023531_870900/selection.json'
selected_weights = SELECTED_ADAPTER / 'adapter_model.safetensors'
assert hashlib.sha256(selected_weights.read_bytes()).hexdigest() == (
    'b1c36a045e164df0efa6a52cbc30cbc4a9e0b01d5fd8359b5a14a5ad73ebf615')
for name, expected in (
    ('m1.jsonl', '4903f4ae527d75999cd95ff6cc51902687966e078db5fc6cc9a3ad8d2ca043b2'),
    ('m1.jsonl.manifest.json', 'a043b5b6285b98cabb020681af2db759058d1bf4643dec708a5fe0c267759ee0'),
):
    assert hashlib.sha256((VALIDATION_JOB / name).read_bytes()).hexdigest() == expected
command = [sys.executable, '-m', 'scripts.freeze_m1_selection',
    '--adapter', str(SELECTED_ADAPTER),
    '--model-config', str(PREFERENCE_REPO / 'configs/model/qwen25_vl_3b.yaml'),
    '--annotations', str(canonical), '--expected-sha256', canonical_sha,
    '--epoch-result', str(FINISHED_M1 / 'training/training_result.json'),
    '--trainer-state', str(FINISHED_M1 / 'training/trainer_state.json'),
    '--loss-selection', str(LOSS_SELECTION),
    '--validation-predictions', str(VALIDATION_JOB / 'm1.jsonl'),
    '--validation-manifest', str(VALIDATION_JOB / 'm1.jsonl.manifest.json'),
    '--validation-annotations', str(validation_snapshot), '--output', str(FROZEN_M1)]
subprocess.run(command, cwd=PREFERENCE_REPO, env=PREFERENCE_ENV, check=True)
FROZEN_M1_SHA = hashlib.sha256(FROZEN_M1.read_bytes()).hexdigest()
print('M1 frozen:', FROZEN_M1, FROZEN_M1_SHA)
print('Source:', PREFERENCE_RELEASE, '| diagnostic review: parallel; no labels invented.')


In [ ]:
# Generate train-only candidates; do not start M2/M3.
from importlib import metadata as candidate_packages

assert 'FROZEN_M1_SHA' in globals(), 'Run the M1 freeze cell first.'
assert FROZEN_M1_SHA == 'db1f26a79a025a2422c9768a8231cfd00dbe4d053683973a216728c551b1d534'
assert hashlib.sha256(FROZEN_M1.read_bytes()).hexdigest() == FROZEN_M1_SHA
assert torch.cuda.is_available(), 'No CUDA GPU: candidate generation has not started.'
pool = ROOT / 'preference_local/question_pool_v1'
assert pool.is_dir(), 'Restore the existing frozen pool; do not silently create another.'
for name, expected in (
    ('questions.jsonl', '4bedfa0cf41ad84a5552ea6b923a85d9a4956fae4c9969f64bc54844d0b2d1ab'),
    ('manifest.json', 'd3c74ac0be475952c7069b573478980af259f1cea30a0bbd0859f6af4a018262'),
):
    assert hashlib.sha256((pool / name).read_bytes()).hexdigest() == expected, name
expected_candidate_versions = {'torch': '2.11.0+cu130', 'transformers': '4.57.6',
    'peft': '0.21.0', 'bitsandbytes': '0.50.2', 'pillow': '11.3.0'}

def installed_candidate_versions():
    versions = {}
    for name in expected_candidate_versions:
        try:
            versions[name] = candidate_packages.version(name)
        except candidate_packages.PackageNotFoundError:
            versions[name] = None
    return versions

versions = installed_candidate_versions()
if versions != expected_candidate_versions:
    print('Restoring pinned inference packages:', versions, flush=True)
    if versions['torch'] != expected_candidate_versions['torch']:
        subprocess.run([sys.executable, '-m', 'pip', 'install',
            'torch==2.11.0+cu130', 'torchvision==0.26.0+cu130',
            '--index-url', 'https://download.pytorch.org/whl/cu130'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install',
        *[f'{name}=={version}' for name, version in expected_candidate_versions.items()
          if name != 'torch']], check=True)
    assert installed_candidate_versions() == expected_candidate_versions
    raise RuntimeError('Packages restored. Restart Python, then run cells 1, 9, 10 only.')
assert torch.__version__ == expected_candidate_versions['torch'], 'Restart Python first.'
print('Inference packages:', versions, '| GPU:', torch.cuda.get_device_name(0), flush=True)
WORK_DATA = Path('/content/vqa_work/vivqax_image_disjoint_v1')
if not WORK_DATA.exists():
    subprocess.run([sys.executable, '-m', 'scripts.verify_coco_bundle', str(BUNDLE),
        '--extract-to', str(WORK_DATA)], cwd=PREFERENCE_REPO, env=PREFERENCE_ENV, check=True)
assert hashlib.sha256((WORK_DATA / 'metadata/all.jsonl').read_bytes()).hexdigest() == canonical_sha
CANDIDATE_JOB = ROOT / 'preference_local/candidates_m1_3683_pool2000_v1'
command = [sys.executable, '-u', '-m', 'scripts.generate_preference_candidates',
    '--selection', str(pool), '--annotations', str(WORK_DATA / 'metadata/all.jsonl'),
    '--expected-sha256', canonical_sha, '--image-root', str(WORK_DATA),
    '--adapter', str(SELECTED_ADAPTER), '--m1-selection', str(FROZEN_M1),
    '--m1-selection-sha256', FROZEN_M1_SHA, '--output-dir', str(CANDIDATE_JOB),
    '--count', '2000', '--seed', '42', '--load-in-4bit']
if CANDIDATE_JOB.exists():
    command.append('--resume')
CANDIDATE_JOB.mkdir(parents=True, exist_ok=True)
# mkdir above only reserves the log directory; the exporter explicitly resumes it.
if '--resume' not in command:
    command.append('--resume')
print('M1-3683: 2.000 train candidates, pending review. Output:', CANDIDATE_JOB, flush=True)
with (CANDIDATE_JOB / 'console.log').open('a', encoding='utf-8') as log:
    with subprocess.Popen(command, cwd=PREFERENCE_REPO, env=PREFERENCE_ENV,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8') as process:
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end='', flush=True)
        assert process.wait() == 0, 'Generation stopped; inspect the saved log before resuming.'
summary = json.loads((CANDIDATE_JOB / 'summary.json').read_text(encoding='utf-8'))
assert summary['n'] == 2000 and summary['status'] == 'generated_pending_review'
print('Candidates saved. Next: review images and freeze accepted/chosen-only; not DPO yet.')


## Lịch sử — không chạy lại

Ô 12–19 giữ kiểm token, lệnh train các lượt trước, đánh giá checkpoint-800, sinh ứng viên và kiểm checkpoint-1800. Giữ nguyên để truy vết; không thuộc luồng hiện tại.

Trạng thái mới nhất ở đầu notebook và trong `TIEN_DO.md`.


In [ ]:
import platform
import random

P0_PASSED = False
RUN_ROOT = ROOT / 'runs'
completed_execution = ROOT / 'training_runs/sft-full_20261002_113657_805762/execution.json'
if completed_execution.is_file():
    previous = json.loads(completed_execution.read_text(encoding='utf-8'))
    recorded_root = Path(previous['paths']['run_root'])
    if not (RUN_ROOT / 'token_full_M1_v1.json').is_file():
        RUN_ROOT = recorded_root
    for name in ('token_full_M1_v1.json', 'm1_environment_v1.json'):
        if not (RUN_ROOT / name).is_file():
            raise FileNotFoundError(f'Missing saved M1 preflight: {RUN_ROOT / name}; do not rescan')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
TOKEN_REPORT = RUN_ROOT / 'token_full_M1_v1.json'
ANNOTATIONS = WORK_DATA / 'metadata/all.jsonl'
PROFILE_CONFIG = REPO / 'configs/training/sft_readiness_full.yaml'
MODEL_CONFIG = REPO / 'configs/model/qwen25_vl_3b.yaml'
if not TOKEN_REPORT.exists():
    command = [
        sys.executable, '-u', str(REPO / 'scripts/profile_sft_tokens.py'),
        '--annotations', str(ANNOTATIONS), '--image-root', str(WORK_DATA),
        '--config', str(PROFILE_CONFIG), '--model-config', str(MODEL_CONFIG),
        '--validation-count', '256', '--output', str(TOKEN_REPORT),
    ]
    with (RUN_ROOT / 'token_full_M1_v1.log').open('a', encoding='utf-8') as log:
        log.write('\n--- Token scan started ---\n')
        with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            if process.wait():
                raise RuntimeError('Token scan failed; inspect token_full_M1_v1.log')
report = json.loads(TOKEN_REPORT.read_text(encoding='utf-8'))
assert report['status'] == 'passed', report
for key, path in [('annotations_sha256', ANNOTATIONS),
                  ('training_config_sha256', PROFILE_CONFIG),
                  ('model_config_sha256', MODEL_CONFIG)]:
    assert report[key] == hashlib.sha256(path.read_bytes()).hexdigest(), key
assert report['sample_size'] == report['train_records'] == 29459
assert report['validation_sample_size'] == 256
assert report['summary']['samples'] == 29715
assert report['summary']['over_max_length'] == 0, report['summary']
rows = [json.loads(line) for line in
        ANNOTATIONS.read_text(encoding='utf-8').splitlines() if line.strip()]
validation = [row for row in rows if row['split'] == 'validation']
random.Random(42).shuffle(validation)
assert report['validation_question_ids'] == [row['question_id'] for row in validation[:256]]
versions = {name: importlib.metadata.version(name) for name in
            ('torch', 'transformers', 'peft', 'accelerate', 'bitsandbytes', 'pillow')}
assert versions['transformers'] == '4.57.6'
for name in ('torch', 'transformers', 'pillow'):
    assert report['versions'][name] == versions[name], name
environment = {
    'source_release': SOURCE_RELEASE, 'source_sha256': EXPECTED_SOURCE_SHA256,
    'python': platform.python_version(), 'versions': versions,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
lock_path = RUN_ROOT / 'm1_environment_v1.json'
if lock_path.exists():
    saved_environment = json.loads(lock_path.read_text(encoding='utf-8'))
    assert saved_environment == environment, 'Environment changed; inspect before resume'
else:
    with lock_path.open('x', encoding='utf-8') as handle:
        json.dump(environment, handle, indent=2)
P0_PASSED = True
print('P0 PASSED:', TOKEN_REPORT)
print(json.dumps(report['summary'], ensure_ascii=False, indent=2))
print('Environment:', lock_path)


In [ ]:
# Recover the saved checkpoint; the previous VM has ended.
assert P0_PASSED, 'P0 chưa hoàn tất'
resume_run = ROOT / 'runs/m1/sft-full_20261006_132346_065189'
execution = json.loads((resume_run / 'execution.json').read_text(encoding='utf-8'))
assert execution['stage'] == 'sft-full', execution
assert execution['status'] in {'running', 'failed', 'interrupted_or_failed'}, execution
checkpoint = resume_run / 'training/checkpoint-3300'
state = json.loads((checkpoint / 'trainer_state.json').read_text(encoding='utf-8'))
assert state['global_step'] == 3300 and state['max_steps'] == 3683, state
assert state['num_train_epochs'] == 1, state
print('Saved M1:', checkpoint, '| target: 3683 (one epoch)', flush=True)
print('Validation losses:', [(row['step'], row['eval_loss'])
      for row in state['log_history'] if 'eval_loss' in row][-4:], flush=True)
EPOCH_LAUNCHER_SHA = 'c9646355195f940e9cac5a6d63818f6df4c7da591085cea773bc01534ed8256d'
launcher = ROOT / 'source' / f'resume_pinned_m1_{EPOCH_LAUNCHER_SHA[:12]}.py'
assert launcher.is_file(), f'Missing epoch launcher: {launcher}'
assert hashlib.sha256(launcher.read_bytes()).hexdigest() == EPOCH_LAUNCHER_SHA
command = [sys.executable, '-u', str(launcher), '--source', str(REPO),
           '--environment', str(REPO / 'configs/environments/colab_full.example.yaml'),
           '--checkpoint', str(checkpoint), '--output-root', str(ROOT / 'runs/m1'),
           '--stop-after-step', '3683', '--budget-protocol', 'vqa_nle_m1_epoch1_v1']
if execution['status'] == 'running':
    command.append('--recover-interrupted')
subprocess.run(command + ['--check-only'], cwd=REPO, check=True)
assert torch.cuda.is_available(), 'A GPU is required to continue M1.'
with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('Epoch continuation stopped; inspect runs/m1 before retrying.')


In [ ]:
# Recover the interrupted run from its last saved checkpoint.
assert P0_PASSED, 'P0 chưa hoàn tất'
checkpoint = ROOT / 'runs/m1/sft-full_20261004_065122_901248/training/checkpoint-1100'
state = json.loads((checkpoint / 'trainer_state.json').read_text(encoding='utf-8'))
assert state['global_step'] == 1100 and state['max_steps'] == 3683, state
metadata = json.loads((checkpoint.parent / 'run_metadata.json').read_text(encoding='utf-8'))
assert metadata['compute_dtype'] in ('torch.float16', 'torch.bfloat16'), metadata
for name in ('optimizer.pt', 'scheduler.pt', 'rng_state.pth',
             'adapter_config.json', 'adapter_model.safetensors'):
    assert (checkpoint / name).is_file() and (checkpoint / name).stat().st_size > 0, name
if metadata['compute_dtype'] == 'torch.float16':
    scaler = checkpoint / 'scaler.pt'
    assert scaler.is_file() and scaler.stat().st_size > 0, 'Missing FP16 scaler'
print('M1 candidate:', checkpoint, '| step:', state['global_step'])
LAUNCHER_SHA = '8589d917d788f5706077337ee467aef46375a1c2e4fa735980267a5b34061cd3'
OUTPUT_ROOT = ROOT / 'runs/m1'
launcher = ROOT / 'source' / f'resume_pinned_m1_{LAUNCHER_SHA[:12]}.py'
assert launcher.is_file(), f'Missing verified launcher: {launcher}'
assert hashlib.sha256(launcher.read_bytes()).hexdigest() == LAUNCHER_SHA
command = [sys.executable, '-u', str(launcher), '--source', str(REPO),
           '--environment', str(REPO / 'configs/environments/colab_full.example.yaml'),
           '--checkpoint', str(checkpoint), '--output-root', str(OUTPUT_ROOT),
           '--stop-after-step', '1200',
           '--budget-protocol', 'vqa_nle_budget_v2_m1_1200', '--recover-interrupted']
subprocess.run(command + ['--check-only'], cwd=REPO, check=True)
assert torch.cuda.is_available(), 'Checkpoint checked on CPU; a GPU is required to resume.'
with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('Resume failed; inspect runs/m1 before retrying.')


In [ ]:
import hashlib
import os
import subprocess
import sys
import zipfile

import tomllib

EVAL_RELEASE = 'source_20261002_062529_455947'
EVAL_SHA256 = '6b00d2d0297d06f4365248fd2bf63fd31c69808700888219ffd6e7c90283d9fb'
eval_zip = ROOT / 'source/releases' / f'{EVAL_RELEASE}.zip'
assert hashlib.sha256(eval_zip.read_bytes()).hexdigest() == EVAL_SHA256
EVAL_BASE = Path('/content/vqa_eval_sources') / EVAL_RELEASE
EVAL_REPO = EVAL_BASE / 'Vietnamese-VQA-Explanation'
if not EVAL_REPO.exists():
    with zipfile.ZipFile(eval_zip) as archive:
        assert all(name.startswith('Vietnamese-VQA-Explanation/')
                   and '..' not in Path(name).parts for name in archive.namelist())
        archive.extractall(EVAL_BASE)
manifest = json.loads((EVAL_REPO / 'SOURCE_MANIFEST.json').read_text())
for name, digest in manifest.items():
    assert hashlib.sha256((EVAL_REPO / name).read_bytes()).hexdigest() == digest, name
project = tomllib.loads((EVAL_REPO / 'pyproject.toml').read_text())['project']
requirements = project['dependencies'] + project['optional-dependencies']['train']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *requirements], check=True)
EVAL_ENV = os.environ.copy()
EVAL_ENV['PYTHONPATH'] = f'{EVAL_REPO / "src"}:{EVAL_REPO}'
WORK_DATA = Path('/content/vqa_work/vivqax_image_disjoint_v1')
bundle_dir = ROOT / 'datasets/releases/vivqax_image_disjoint_v1'
bundle_bytes = (bundle_dir / 'bundle.json').read_bytes()
assert hashlib.sha256(bundle_bytes).hexdigest() == (
    '46846a81c7b3594d4fc9670ec51019cfd652f1eb169af7978e39dc5a40a55522')
bundle = json.loads(bundle_bytes)
if not WORK_DATA.exists():
    print('Restoring data to the VM; Drive keeps the run outputs.', flush=True)
    subprocess.run([sys.executable, '-u', str(EVAL_REPO / 'scripts/verify_coco_bundle.py'),
                    str(bundle_dir), '--extract-to', str(WORK_DATA)],
                   cwd=EVAL_REPO, env=EVAL_ENV, check=True)
assert len(list((WORK_DATA / 'images').glob('*.jpg'))) == bundle['total_images']
assert hashlib.sha256((WORK_DATA / 'metadata/full_image_checksums.csv').read_bytes()
                      ).hexdigest() == bundle['image_checksums_sha256']
print('Evaluation source:', EVAL_RELEASE, '| images:', bundle['total_images'])


In [ ]:
assert torch.cuda.is_available(), 'Validation inference requires GPU'
image_check = (
    'from scripts.run_m0 import read_jsonl; '
    'from pathlib import Path; '
    'from vqa_explanation.training.image_integrity import verify_record_images; '
    f'p=Path({str(WORK_DATA)!r}); '
    'print(verify_record_images(read_jsonl(p/"metadata/validation.jsonl"), '
    'p, p/"metadata/full_image_checksums.csv"))'
)
subprocess.run([sys.executable, '-c', image_check],
               cwd=EVAL_REPO, env=EVAL_ENV, check=True)
candidate = ROOT / 'training_runs/sft-full_20261002_113657_805762/training/checkpoint-800'
assert json.loads((candidate / 'trainer_state.json').read_text())['global_step'] == 800

def prepare_validation_job(old_job, relocated_job, checkpoint):
    from filelock import FileLock

    from scripts.run_baseline_validation import build_commands
    from scripts.run_m0 import read_jsonl
    from vqa_explanation.utils.environment import load_environment

    environment = load_environment(EVAL_REPO / 'configs/environments/colab_full.example.yaml')
    saved_plan = json.loads((old_job / 'commands.json').read_text())
    if saved_plan == build_commands(environment, checkpoint.resolve(), old_job.resolve()):
        return old_job
    if len(read_jsonl(old_job / 'm0.jsonl')) != 1407:
        raise ValueError('Relocation requires the complete saved M0 export')
    relocated_job.mkdir(parents=True, exist_ok=True)
    with FileLock(str(relocated_job / 'validation.lock'), timeout=0):
        plan_path = relocated_job / 'commands.json'
        plan = build_commands(environment, checkpoint.resolve(), relocated_job.resolve())
        if plan_path.exists():
            if json.loads(plan_path.read_text()) != plan:
                raise ValueError('Relocated job has different commands; preserve it')
        else:
            if list(relocated_job.glob('*.jsonl')):
                raise ValueError('Existing relocated predictions have no job plan')
            with plan_path.open('x', encoding='utf-8') as handle:
                json.dump(plan, handle, indent=2)
        for name in ('m0.jsonl', 'm0.jsonl.manifest.json'):
            payload = (old_job / name).read_bytes()
            destination = relocated_job / name
            if destination.exists():
                if destination.read_bytes() != payload:
                    raise ValueError(f'Relocated M0 differs: {destination}')
            else:
                with destination.open('xb') as handle:
                    handle.write(payload)
            print('Reused M0:', name, hashlib.sha256(payload).hexdigest(), flush=True)
    return relocated_job

sys.path[:0] = [str(EVAL_REPO / 'src'), str(EVAL_REPO)]
VALIDATION_JOB = prepare_validation_job(
    ROOT / 'evaluation_runs/baseline_validation_600_mount_v1',
    ROOT / 'evaluation_local/baseline_validation_800_v1', candidate)
print('Validation job:', VALIDATION_JOB, flush=True)
command = [sys.executable, '-u', '-m', 'scripts.run_baseline_validation',
           '--environment', 'configs/environments/colab_full.example.yaml',
           '--checkpoint', str(candidate), '--output-dir', str(VALIDATION_JOB), '--resume']
with subprocess.Popen(command, cwd=EVAL_REPO, env=EVAL_ENV,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('Validation stopped; inspect the saved log before retrying.')
print('Full validation report:', VALIDATION_JOB / 'full_validation.json')


In [ ]:
# Read completed reports and export the evidence for local analysis.
import hashlib
import json
import zipfile
from pathlib import Path

from google.colab import files

old_job = ROOT / 'evaluation_runs/baseline_validation_600_mount_v1'
new_job = ROOT / 'evaluation_local/baseline_validation_800_v1'
reports = {}
for step, job in ((600, old_job), (800, new_job)):
    report = json.loads((job / 'full_validation.json').read_text())
    assert report['comparable'] and not report['issues'], report
    reports[step] = report
    print('Checkpoint:', step)
    print(json.dumps(report, ensure_ascii=False, indent=2))
assert (old_job / 'm0.jsonl').read_bytes() == (new_job / 'm0.jsonl').read_bytes()

members = {}
for step, job in ((600, old_job), (800, new_job)):
    for name in ('full_validation.json', 'commands.json', 'm0.jsonl',
                 'm0.jsonl.manifest.json', 'm1.jsonl', 'm1.jsonl.manifest.json'):
        members[f'checkpoint_{step}/{name}'] = (job / name).read_bytes()
members['validation.jsonl'] = (WORK_DATA / 'metadata/validation.jsonl').read_bytes()
assert hashlib.sha256(members['validation.jsonl']).hexdigest() == (
    '7f0b2b7d374d386030ddea50bc57d3e8e6c76e257e9fc4d1203bfb331b5f5b6e')
members['checksums.json'] = json.dumps(
    {name: hashlib.sha256(data).hexdigest() for name, data in members.items()},
    indent=2).encode()
review_zip = new_job / 'validation_review_600_800.zip'
if review_zip.exists():
    with zipfile.ZipFile(review_zip) as archive:
        assert set(archive.namelist()) == set(members)
        assert all(archive.read(name) == data for name, data in members.items())
else:
    with zipfile.ZipFile(review_zip, 'x', zipfile.ZIP_DEFLATED) as archive:
        for name, data in members.items():
            archive.writestr(name, data)
print('Review bundle:', review_zip, hashlib.sha256(review_zip.read_bytes()).hexdigest())
files.download(str(review_zip))


In [ ]:
# Generate the first 100 train candidates; no preference labels are assigned.
import hashlib
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import files

PREFERENCE_RELEASE = 'source_20261003_064654_019183'
PREFERENCE_SHA256 = 'e8fed49c688082c91512a7324d587c4c25048674356de91662f0dd6d451a4986'
preference_zip = ROOT / 'source/releases' / f'{PREFERENCE_RELEASE}.zip'
assert hashlib.sha256(preference_zip.read_bytes()).hexdigest() == PREFERENCE_SHA256
PREFERENCE_BASE = Path('/content/vqa_preference_sources') / PREFERENCE_RELEASE
PREFERENCE_REPO = PREFERENCE_BASE / 'Vietnamese-VQA-Explanation'
if not PREFERENCE_REPO.exists():
    with zipfile.ZipFile(preference_zip) as archive:
        assert len(archive.namelist()) == len(set(archive.namelist()))
        assert all(name.startswith('Vietnamese-VQA-Explanation/')
                   and '..' not in Path(name).parts for name in archive.namelist())
        archive.extractall(PREFERENCE_BASE)
source_manifest = json.loads((PREFERENCE_REPO / 'SOURCE_MANIFEST.json').read_text())
for name, digest in source_manifest.items():
    assert hashlib.sha256((PREFERENCE_REPO / name).read_bytes()).hexdigest() == digest, name
PREFERENCE_ENV = os.environ.copy()
PREFERENCE_ENV['PYTHONPATH'] = f'{PREFERENCE_REPO / "src"}:{PREFERENCE_REPO}'
canonical = WORK_DATA / 'metadata/all.jsonl'
canonical_sha = '9750ab17cf6dae74680404e167ab1c29d1e2083c8baa2e25bfafcf14bff29c31'
assert hashlib.sha256(canonical.read_bytes()).hexdigest() == canonical_sha
pool = ROOT / 'preference_local/question_pool_v1'
calibration = ROOT / 'preference_local/calibration_m1_800_100_v1'
if not pool.exists():
    subprocess.run([sys.executable, '-u', '-m', 'scripts.select_preference_questions',
                    '--annotations', str(canonical), '--image-root', str(WORK_DATA),
                    '--output-dir', str(pool), '--count', '2000', '--seed', '42'],
                   cwd=PREFERENCE_REPO, env=PREFERENCE_ENV, check=True)
candidate = ROOT / (
    'training_runs/sft-full_20261002_113657_805762/training/checkpoint-800')
assert json.loads((candidate / 'trainer_state.json').read_text())['global_step'] == 800
command = [sys.executable, '-u', '-m', 'scripts.generate_preference_candidates',
           '--selection', str(pool), '--annotations', str(canonical),
           '--expected-sha256', canonical_sha, '--image-root', str(WORK_DATA),
           '--adapter', str(candidate), '--output-dir', str(calibration),
           '--count', '100', '--seed', '42', '--load-in-4bit']
if calibration.exists():
    command.append('--resume')
print('Generating 100 train candidates from checkpoint-800. Outputs stay on Drive.', flush=True)
subprocess.run(command, cwd=PREFERENCE_REPO, env=PREFERENCE_ENV, check=True)
members = {}
for name in ('inputs.jsonl', 'candidates.jsonl', 'candidates.jsonl.manifest.json', 'summary.json'):
    members[name] = (calibration / name).read_bytes()
for name in ('questions.jsonl', 'manifest.json'):
    members[f'pool/{name}'] = (pool / name).read_bytes()
inputs = [json.loads(line) for line in members['inputs.jsonl'].decode().splitlines()]
for row in inputs:
    members[row['image_path']] = (WORK_DATA / row['image_path']).read_bytes()
members['checksums.json'] = json.dumps(
    {name: hashlib.sha256(data).hexdigest() for name, data in members.items()}, indent=2).encode()
calibration_zip = calibration / 'review_100.zip'
if calibration_zip.exists():
    with zipfile.ZipFile(calibration_zip) as archive:
        assert set(archive.namelist()) == set(members)
        assert all(archive.read(name) == data for name, data in members.items())
else:
    with zipfile.ZipFile(calibration_zip, 'x', zipfile.ZIP_DEFLATED) as archive:
        for name, data in members.items():
            archive.writestr(name, data)
print('Pending review bundle:', calibration_zip,
      hashlib.sha256(calibration_zip.read_bytes()).hexdigest(), flush=True)
files.download(str(calibration_zip))


In [ ]:
# CPU-only integrity check; no training or checkpoint writes.
import hashlib
import importlib.util


def download_audit(path):
    from google.colab import files

    files.download(str(path))

assert 'ROOT' in globals() and ROOT.is_dir(), 'Run cell 1 to mount the project Drive.'
AUDIT_LAUNCHER_SHA = 'c9646355195f940e9cac5a6d63818f6df4c7da591085cea773bc01534ed8256d'
audit_launcher = ROOT / 'source/resume_pinned_m1_c9646355195f.py'
assert (
    hashlib.sha256(audit_launcher.read_bytes()).hexdigest() == AUDIT_LAUNCHER_SHA
), 'Audit launcher changed.'
spec = importlib.util.spec_from_file_location('m1_checkpoint_audit', audit_launcher)
audit_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit_module)

audit_run = ROOT / 'runs/m1/sft-full_20261005_075818_911934'
checkpoint = audit_run / 'training/checkpoint-1800'
execution = json.loads((audit_run / 'execution.json').read_text(encoding='utf-8'))
state = json.loads((checkpoint / 'trainer_state.json').read_text(encoding='utf-8'))
assert execution['stage'] == 'sft-full'
assert state['global_step'] == 1800 and state['max_steps'] == 3683
integrity = audit_module.verify_checkpoint_payloads(checkpoint)
audit_report = {
    'checkpoint': str(checkpoint.relative_to(ROOT)),
    'launcher_sha256': AUDIT_LAUNCHER_SHA,
    'epoch': state['epoch'],
    'best_validation_loss': state['best_metric'],
    'integrity': integrity,
    'scope': 'CPU payload integrity, not a training readiness check.',
}
audit_text = json.dumps(audit_report, ensure_ascii=False, indent=2, sort_keys=True) + '\n'
audit_dir = ROOT / 'reports/checkpoints'
audit_dir.mkdir(parents=True, exist_ok=True)
audit_path = audit_dir / 'm1_sft-full_20261005_075818_911934_checkpoint-1800.json'
if audit_path.exists():
    assert (
        audit_path.read_text(encoding='utf-8') == audit_text
    ), 'Existing audit differs; inspect without overwriting.'
else:
    with audit_path.open('x', encoding='utf-8') as stream:
        stream.write(audit_text)
print('Checkpoint payload passed on CPU:', json.dumps(audit_report, ensure_ascii=False))
print('Saved audit:', audit_path, hashlib.sha256(audit_path.read_bytes()).hexdigest())
download_audit(audit_path)
